In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from algbench import read_as_pandas
from matplotlib.ticker import PercentFormatter
from tspn_bnb2.misc.paper_style import FULLWIDEFIGURE, init_params
from tspn_bnb2.misc.plotting import cactus_plot
from tspn_bnb2.schemas import AnnotatedInstance, AnnotatedSolution

init_params()

In [ ]:
map_root_name = {
    "LongestEdgePlusFurthestSite": "LEFP",
    "RandomRoot": "Random",
    "OrderRoot": "CHR",
    "LongestPair": "LE",
}

In [ ]:
def read_row(row):
    instance = AnnotatedInstance.model_validate_json(
        row["parameters"]["args"]["kwargs"]["instance_json"]
    )
    solution = AnnotatedSolution.model_validate_json(row["result"]["solution"])
    return {
        "solution": solution,
        "gap": solution.relative_gap,
        "is_optimal": solution.is_optimal,
        "instance_type": instance.derive_instance_type(),
        "instance_name": row["parameters"]["args"]["kwargs"]["instance_name"],
        "instance": instance,
        "solve_time": row["result"]["solve_time"],
        "root_strategy": map_root_name[row["parameters"]["args"]["alg_params"]["root"]],
        "num_explored": solution.statistics.get("num_explored", None),
    }


benchmark = read_as_pandas("results_root_strategy", read_row)
benchmark = benchmark.sort_values(by=["root_strategy"])
print(
    "loaded",
    len(benchmark),
    "runs",
    benchmark["root_strategy"].unique(),
    len(benchmark["instance_name"].unique()),
)

In [ ]:
result = (
    benchmark.groupby(["instance_name", "root_strategy"])["is_optimal"]
    .all()  # True if all rows per (instance, strategy) are True
    .groupby("instance_name")
    .all()  # True if all strategies for that instance are True
)

valid_instances = result[result].index.tolist()
print(len(valid_instances), "solved by all solvers")

In [ ]:
# validate that solutions are correct.
n_checks = 0
for _, row in benchmark.iterrows():
    solution = row["solution"]
    if solution is None:
        continue
    same_instance = benchmark[benchmark["instance_name"] == row["instance_name"]]

    assert solution.check_feasibility(row["instance"], eps=0.01)
    for _, other in same_instance.iterrows():
        if other["solution"] is None:
            continue
        check = solution.plausibility_check(other["solution"], eps=0.01)
        assert check["valid"], (
            f"Check failed for {row['instance_name']} {check}"
            f" {other['solution']} {row['root_strategy']}"
        )
        n_checks += 1

print(n_checks, "checks succeeded")

In [ ]:
fig, ax = plt.subplots(figsize=FULLWIDEFIGURE)

sns.boxplot(
    benchmark,
    x="root_strategy",
    y="solve_time",
    hue="instance_type",
    hue_order=["OSM", "tessellation", "random"],
)

xticks = list(ax.get_xticks())
xticklabels = []

for label in ax.get_xticklabels():
    solutions_for_label = benchmark[(benchmark["root_strategy"] == label.get_text())]
    optimal_percentage = len(solutions_for_label[solutions_for_label["is_optimal"]]) / len(
        solutions_for_label
    )

    label = label.get_text() + f"\n[{round(optimal_percentage * 100, 2)}\\%]"
    xticklabels.append(label)

ax.set_xticks(xticks)
ax.set_xticklabels(xticklabels)

ax.set_title("lower is better")
ax.set_xlabel("root node strategy")
ax.set_ylabel("runtime [s]")

leg = ax.legend(loc="upper right")
leg.set_title(None)

fig.savefig("../plots/rq1_root_node_selection/runtimes.pdf", bbox_inches="tight")

In [ ]:
from matplotlib.lines import Line2D

fig, axs = plt.subplots(ncols=2, figsize=FULLWIDEFIGURE)

ax = axs[0]
ax2 = axs[1]

colors = {v: f"C{i}" for i, v in enumerate(benchmark["root_strategy"].unique())}

benchmark_solved = benchmark[benchmark["is_optimal"]]
instance_types = ["random", "tessellation", "OSM"]
linestyles = [
    "-",  # solid
    (0, (5, 2)),  # long dashes
    (0, (1, 2)),  # fine dots
]
instance_type_handles = []
labels = None
handles = None
for i, (instance_type, linestyle) in enumerate(zip(instance_types, linestyles)):
    cactus_plot(
        benchmark_solved[benchmark_solved["instance_type"] == instance_type],
        ax=ax,
        instance_column="instance_name",
        strategy_column="root_strategy",
        runtime_column="solve_time",
        flat_line_to=benchmark["solve_time"].max(),
        colors=colors,
        linestyle=linestyle,
    )

    if i == 0:
        handles, labels = ax.get_legend_handles_labels()

    instance_type_handles.append(
        Line2D([], [], color="black", linestyle=linestyle, label=instance_type)
    )


caption = Line2D([], [], linestyle="none", label="instances")
instance_type_handles.insert(0, caption)

# Combine legends: first strategies, then simplified info
fig.legend(
    handles=handles + instance_type_handles,
    labels=labels + [h.get_label() for h in instance_type_handles],
    loc="outside left center",
    ncol=1,
)

bm = benchmark[~benchmark["is_optimal"]]

sns.boxplot(
    data=bm,
    x="root_strategy",
    y="gap",
    hue="root_strategy",
    dodge=False,
    ax=ax2,
)
ax2.set_xlabel("")
ax2.set_ylabel("gap [\\%]")
ax2.yaxis.set_major_formatter(PercentFormatter(1.0))

xticks = list(ax2.get_xticks())
xticklabels = []


def to_percentage(x):
    return round(x * 100, 2)


for label in ax2.get_xticklabels():
    percentages = {}
    solutions_for_label = benchmark[(benchmark["root_strategy"] == label.get_text())]
    if len(solutions_for_label) == 0:
        continue

    percentages = len(solutions_for_label[solutions_for_label["is_optimal"]]) / len(
        solutions_for_label
    )

    label = label.get_text() + f"\n\\tiny[{to_percentage(percentages)}\\%]"
    xticklabels.append(label)

ax2.set_xticks(xticks)
ax2.set_xticklabels(xticklabels)

# ax.axhline(y=benchmark["instance_name"].nunique(), linestyle="--", color="gray")
# ax.set_ylim(0, benchmark["instance_name"].nunique()+5)


ax.set_title("higher is better")
ax2.set_title("lower is better")
fig.subplots_adjust(left=0.25, wspace=0.43)

fig.savefig("../plots/rq1_root_node_selection/cactus_and_gaps.pdf")

In [ ]:
from matplotlib.lines import Line2D

fig, axs = plt.subplots(ncols=2, figsize=FULLWIDEFIGURE)

ax = axs[0]
ax2 = axs[1]

colors = {v: f"C{i}" for i, v in enumerate(benchmark["root_strategy"].unique())}

benchmark_solved = benchmark
instance_types = ["random", "tessellation", "OSM"]
linestyles = [
    "-",  # solid
    (0, (5, 2)),  # long dashes
    (0, (1, 2)),  # fine dots
]
instance_type_handles = []
labels = None
handles = None
for i, (instance_type, linestyle) in enumerate(zip(instance_types, linestyles)):
    cactus_plot(
        benchmark_solved[benchmark_solved["instance_type"] == instance_type],
        ax=ax,
        instance_column="instance_name",
        strategy_column="root_strategy",
        runtime_column="gap",
        flat_line_to=benchmark["gap"].max(),
        colors=colors,
        linestyle=linestyle,
    )

    if i == 0:
        handles, labels = ax.get_legend_handles_labels()

    instance_type_handles.append(
        Line2D([], [], color="black", linestyle=linestyle, label=instance_type)
    )


caption = Line2D([], [], linestyle="none", label="instances")
instance_type_handles.insert(0, caption)

# Combine legends: first strategies, then simplified info
fig.legend(
    handles=handles + instance_type_handles,
    labels=labels + [h.get_label() for h in instance_type_handles],
    loc="outside left center",
    ncol=1,
)

bm = benchmark[benchmark["instance_name"].isin(valid_instances)].copy()
bm["average_time_per_node"] = bm["solve_time"] / bm["num_explored"]
sns.boxplot(
    data=bm,
    x="root_strategy",
    y="average_time_per_node",
    hue="root_strategy",
    dodge=False,
    ax=ax2,
)

ax.xaxis.set_major_formatter(PercentFormatter(1.0))

xticks = list(ax2.get_xticks())
xticklabels = []


def to_percentage(x):
    return round(x * 100, 2)


for label in ax2.get_xticklabels():
    percentages = {}
    solutions_for_label = benchmark[(benchmark["root_strategy"] == label.get_text())]
    if len(solutions_for_label) == 0:
        continue

    percentages = len(solutions_for_label[solutions_for_label["is_optimal"]]) / len(
        solutions_for_label
    )

    label = label.get_text() + f"\n\\tiny[{to_percentage(percentages)}\\%]"
    xticklabels.append(label)

ax2.set_xticks(xticks)
ax2.set_xticklabels(xticklabels)

# ax.axhline(y=benchmark["instance_name"].nunique(), linestyle="--", color="gray")
# ax.set_ylim(0, benchmark["instance_name"].nunique()+5)


ax.set_title("higher is better")
ax.set_xlim(0.01, benchmark["gap"].max())
ax.set_xlabel(r"gap [\%]")

ax2.set_title("lower is better")
ax2.set_xlabel("")
ax2.set_ylabel(r"avg. time per node [s]")

fig.subplots_adjust(left=0.25, wspace=0.43)

fig.savefig("../plots/rq1_root_node_selection/cactus_and_gaps.pdf", facecolor="white")

In [ ]:
from matplotlib.lines import Line2D

fig, axs = plt.subplots(ncols=2, figsize=FULLWIDEFIGURE)

colors = {v: f"C{i}" for i, v in enumerate(benchmark["root_strategy"].unique())}

benchmark_solved = benchmark
instance_types = ["random", ["tessellation", "OSM"]]

for i, (ax, instance_type) in enumerate(zip(axs, instance_types)):
    if isinstance(instance_type, list):
        solved = benchmark_solved[benchmark_solved["instance_type"].isin(instance_type)]
    else:
        solved = benchmark_solved[benchmark_solved["instance_type"] == instance_type]

    max_gap_for_type = solved["gap"].max()
    cactus_plot(
        solved,
        ax=ax,
        instance_column="instance_name",
        strategy_column="root_strategy",
        runtime_column="gap",
        flat_line_to=max_gap_for_type,
        colors=colors,
        linestyle="-",
    )

    if i != 0:
        ax.legend().remove()
        ax.set_ylabel("")

    ax.set_title(", ".join(instance_type) if isinstance(instance_type, list) else instance_type)
    ax.set_xlim(0.01, max_gap_for_type)
    ax.set_xlabel(r"gap [\%]")
    ax.xaxis.set_major_formatter(PercentFormatter(1.0))


axs[0].legend()

fig.suptitle("higher is better")
fig.subplots_adjust(top=0.8)
fig.savefig("../plots/rq1_root_node_selection/cactus_by_instance_type.pdf", facecolor="white")